In [1]:
import sys
from scipy.io import mmread
import os
import glob
import pandas as pd
import numpy as np
#from pandas_ods_reader import read_ods
from copy import deepcopy
import pprint
import json
import re
from datetime import datetime
import logging
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import HuberRegressor
from sklearn import preprocessing
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.distance import pdist
from scipy.spatial.distance import squareform
from sklearn.manifold import TSNE
from sklearn import metrics
from sklearn.cluster import DBSCAN
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
from collections import Counter
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage
import harmonypy as hm
from matplotlib.cm import ScalarMappable
from datetime import date
import mpld3
import hvplot.pandas
import holoviews as hv
from holoviews import opts
import panel as pn
import bokeh
from bokeh.resources import INLINE
from adjustText import adjust_text
from scipy.stats import mannwhitneyu, false_discovery_control, wilcoxon
import pygwalker as pyg
import matplotlib as mpl 

import dimorph_processing as dp
import cell_comparison as cc
import sex_stats as ss

today = str(date.today())
%matplotlib notebook
%load_ext autoreload
%autoreload 2

#change matplotlib font type to make compatibile with illustrator
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

In [302]:
#cluster_fn = 'Vglut2-6-Otp-Zic5'
run = '311224_run'
cell_class = 'gaba'
delta_data_folder = '/bigdata/isaac/'+ cell_class + '_files/sex_stats/'+run+'/gene_delta_plots/data/'
#test_df = pd.read_json(delta_data_folder + cell_class + '_expr_mlog_df_c_' +cluster_fn +'.json')

In [303]:
# Change the current working directory to the specified path
os.chdir(delta_data_folder)

# Get a list of all files in the directory that include '_expr_mlog_df_c_'
files = [f for f in os.listdir() if '_expr_mlog_df_c_' in f]

# Print the list of files
print(files)

['gaba_expr_mlog_df_c_GABA-13-Fign-Lrpprc.json', 'gaba_expr_mlog_df_c_GABA-16-Lmo1-Chn2.json', 'gaba_expr_mlog_df_c_GABA-29-Lpyd1-Unc13c.json', 'gaba_expr_mlog_df_c_GABA-33-Cartpt-Unc13c.json', 'gaba_expr_mlog_df_c_GABA-24-Prlr-St18.json', 'gaba_expr_mlog_df_c_GABA-20-Meis2-Foxp2.json', 'gaba_expr_mlog_df_c_GABA-22-Col23a1-Hs3st4.json', 'gaba_expr_mlog_df_c_GABA-2-Sst-Npy-Maf.json', 'gaba_expr_mlog_df_c_GABA-17-Isl1-Pou3f2.json', 'gaba_expr_mlog_df_c_GABA-21-Meis2-Nwd2.json', 'gaba_expr_mlog_df_c_GABA-3-Sst-Npy-Chodl.json', 'gaba_expr_mlog_df_c_GABA-15-Igsf1-Zfhx3.json', 'gaba_expr_mlog_df_c_GABA-27-Greb1-Dlk1.json', 'gaba_expr_mlog_df_c_GABA-6-Hapln1-Cryab.json', 'gaba_expr_mlog_df_c_GABA-19-Meis2-Tshz1.json', 'gaba_expr_mlog_df_c_GABA-25-Oprk1-Trhde.json', 'gaba_expr_mlog_df_c_GABA-28-Lypd1-Satb1.json', 'gaba_expr_mlog_df_c_GABA-11-Pax6-Calca.json', 'gaba_expr_mlog_df_c_GABA-32-Cartpt-Dkk2.json', 'gaba_expr_mlog_df_c_GABA-10-Pax6-Npnt.json', 'gaba_expr_mlog_df_c_GABA-4-Moxd1-Pvalb.js

In [25]:
x = files[0].split('_')
x[-1].split('.')[0]

'GABA-13-Fign-Lrpprc'

In [7]:
#cluster_fn = 'Vglut2-6-Otp-Zic5'
run = '040325_run'
cell_class = 'Nonneuronal'
delta_data_folder = '/bigdata/isaac/'+ cell_class + '_files/sex_stats/'+run+'/gene_delta_plots/data/'
#test_df = pd.read_json(delta_data_folder + cell_class + '_expr_mlog_df_c_' +cluster_fn +'.json')

#hormone receptive markers to consider
dp_markers = ['Esr1','Esr2','Ar','Pgr']

color_mapping = {
    'B_f': 'blue',
    'B_m': 'orange',
    'N_m': 'red',
    'N_f': 'green'
}

# Change the current working directory to the specified path
os.chdir(delta_data_folder)

# Get a list of all files in the directory that include '_expr_mlog_df_c_'
files = [f for f in os.listdir() if '_expr_mlog_df_c_' in f]
print (files)

# Define the new directory name
hbp_dir = 'hormone_bar_plots'
# Create the new directory
hbp_dir_path = os.path.join('/bigdata/isaac/'+ cell_class + '_files/sex_stats/'+run, hbp_dir)
os.makedirs(hbp_dir_path, exist_ok=True)
for f in files:
    #print (f)
    
    #get cluster fn
    x = f.split('_')
    fn = x[-1].split('.')[0]
    #read in mean log data
    test_df = pd.read_json(f)
    dp_markers_in_test_df = []
    for g in dp_markers:
        if g in test_df.index:
            dp_markers_in_test_df.append(g)
    df = test_df.loc[dp_markers_in_test_df]
    #colors = [color_mapping.get(col, 'black') for col in df.columns]
    df.plot(kind='bar', width=0.8, figsize=(8, 4), colormap='Set2')
    plt.ylabel('Mean Log Expression')
    plt.title(fn + ' Mean Log Expression')
    plt.xticks(rotation=0)
    plt.legend(title='Group')
    plt.tight_layout()
    plt.savefig(hbp_dir_path + '/'+ cell_class + '_hbp_' + fn + '.png')
    plt.show()
    
    

['Nonneuronal_expr_mlog_df_c_Nonneuronal-10-Endothelial.json', 'Nonneuronal_expr_mlog_df_c_Nonneuronal-1-OL_1.json', 'Nonneuronal_expr_mlog_df_c_Nonneuronal-11-VSM.json', 'Nonneuronal_expr_mlog_df_c_Nonneuronal-5-Astro_gfap.json', 'Nonneuronal_expr_mlog_df_c_Nonneuronal-6-Astro.json', 'Nonneuronal_expr_mlog_df_c_Nonneuronal-3-OL_3.json', 'Nonneuronal_expr_mlog_df_c_Nonneuronal-9-Microglia.json']


<IPython.core.display.Javascript object>

IndexError: index 0 is out of bounds for axis 0 with size 0

In [305]:
files[4]

'gaba_expr_mlog_df_c_GABA-24-Prlr-St18.json'

In [307]:
test_df

,N_f,B_f,N_m,B_m
0610007P14Rik,0.627974,0.356680,0.562874,0.322593
0610009B22Rik,0.372232,0.221154,0.382525,0.224377
0610009L18Rik,0.141602,0.057692,0.111421,0.116675
0610009O20Rik,0.131893,0.126634,0.142105,0.070270
0610010F05Rik,0.269465,0.130625,0.345683,0.185621
...,...,...,...,...
Zyg11b,0.466607,0.601244,0.655332,0.445089
Zzef1,0.205533,0.320575,0.434365,0.270782
Zzz3,0.338172,0.410382,0.377176,0.335647
l7Rn6,0.529221,0.353412,0.551453,0.349620


<IPython.core.display.Javascript object>

In [21]:
print (hbp_dir_path)

/bigdata/isaac/Vglut1_files/sex_stats/311224_run/hormone_bar_plots


In [22]:
test_df

,N_f,B_f,N_m,B_m
0610007P14Rik,0.400000,0.351294,0.507724,0.222222
0610009B22Rik,0.200000,0.131068,0.181818,0.620551
0610010F05Rik,0.000000,0.310395,0.379084,0.333333
0610011F06Rik,0.916993,0.188969,0.090909,0.287218
0610012G03Rik,1.233985,0.550044,1.232931,1.322988
...,...,...,...,...
Zzef1,0.400000,0.160305,0.507724,0.444444
Zzz3,0.400000,0.208701,0.325906,0.767432
l7Rn6,0.200000,0.225839,0.545455,0.574436
mt-Atp8,1.464386,0.730252,1.588103,1.746741


Get raw (gene sum normalized) expression data and metadata, build trend lines for each group

In [205]:
cell_class = 'gaba'
delta_data_folder = '/bigdata/isaac/'+ cell_class + '_files/sex_stats/'+run+'/gene_delta_plots/data/'

'gaba'

['gaba_expr_raw_df_c_GABA-2-Sst-Npy-Maf.json', 'gaba_expr_raw_df_c_GABA-26-Col18a1-Jsrp1.json', 'gaba_expr_raw_df_c_GABA-30-Cbln4-Zbtb20.json', 'gaba_expr_raw_df_c_GABA-7-Vip-Crh.json', 'gaba_expr_raw_df_c_GABA-28-Lypd1-Satb1.json', 'gaba_expr_raw_df_c_GABA-29-Lpyd1-Unc13c.json', 'gaba_expr_raw_df_c_GABA-21-Meis2-Nwd2.json', 'gaba_expr_raw_df_c_GABA-4-Moxd1-Pvalb.json', 'gaba_expr_raw_df_c_GABA-13-Fign-Lrpprc.json', 'gaba_expr_raw_df_c_GABA-18-Meis2-Dach1.json', 'gaba_expr_raw_df_c_GABA-10-Pax6-Npnt.json', 'gaba_expr_raw_df_c_GABA-19-Meis2-Tshz1.json', 'gaba_expr_raw_df_c_GABA-12-Sncg-Reln.json', 'gaba_expr_raw_df_c_GABA-3-Sst-Npy-Chodl.json', 'gaba_expr_raw_df_c_GABA-33-Cartpt-Unc13c.json', 'gaba_expr_raw_df_c_GABA-17-Isl1-Pou3f2.json', 'gaba_expr_raw_df_c_GABA-5-Moxd1-Vwc2.json', 'gaba_expr_raw_df_c_GABA-9-Dab1-Myh7.json', 'gaba_expr_raw_df_c_GABA-20-Meis2-Foxp2.json', 'gaba_expr_raw_df_c_GABA-31-Fign-Foxp2.json', 'gaba_expr_raw_df_c_GABA-27-Greb1-Dlk1.json', 'gaba_expr_raw_df_c_GABA

['gaba_expr_raw_df_c_GABA-1-Sst-Npy-Spon1.json', 'gaba_expr_raw_df_c_GABA-2-Sst-Npy-Maf.json', 'gaba_expr_raw_df_c_GABA-3-Sst-Npy-Chodl.json', 'gaba_expr_raw_df_c_GABA-4-Moxd1-Pvalb.json', 'gaba_expr_raw_df_c_GABA-5-Moxd1-Vwc2.json', 'gaba_expr_raw_df_c_GABA-6-Hapln1-Cryab.json', 'gaba_expr_raw_df_c_GABA-7-Vip-Crh.json', 'gaba_expr_raw_df_c_GABA-8-Htr3a-Rgs12.json', 'gaba_expr_raw_df_c_GABA-9-Dab1-Myh7.json', 'gaba_expr_raw_df_c_GABA-10-Pax6-Npnt.json', 'gaba_expr_raw_df_c_GABA-11-Pax6-Calca.json', 'gaba_expr_raw_df_c_GABA-12-Sncg-Reln.json', 'gaba_expr_raw_df_c_GABA-13-Fign-Lrpprc.json', 'gaba_expr_raw_df_c_GABA-14-Isl1-Gal.json', 'gaba_expr_raw_df_c_GABA-15-Igsf1-Zfhx3.json', 'gaba_expr_raw_df_c_GABA-16-Lmo1-Chn2.json', 'gaba_expr_raw_df_c_GABA-17-Isl1-Pou3f2.json', 'gaba_expr_raw_df_c_GABA-18-Meis2-Dach1.json', 'gaba_expr_raw_df_c_GABA-19-Meis2-Tshz1.json', 'gaba_expr_raw_df_c_GABA-20-Meis2-Foxp2.json', 'gaba_expr_raw_df_c_GABA-21-Meis2-Nwd2.json', 'gaba_expr_raw_df_c_GABA-22-Col23a

In [181]:
sorted_metadata_files[12]

'gaba_metadata_df_c_GABA-13-Fign-Lrpprc.json'

In [182]:
raw_df = pd.read_json(sorted_raw_files[12])
raw_df

,CCCGAAGCATCAACCA-1_10X35_2,TCTCCGACAAACCACT-1_10X38_2,TCAAGTGAGAGCAACC-1_10X35_1,CCGTTCACACTCCTGT-1_10X35_1,TCCTCCCCAATGCTCA-1_10X35_2,ATCCGTCAGTATGACA-1_10X35_2,ATCTTCATCCAGCTCT-1_10X35_1,CCCGGAATCCATATGG-1_10X35_2,GACAGCCTCGACACTA-1_10X38_1,CACTTCGCATCGAACT-1_10X38_1,...,AGAAATGGTTTATGCG-1_10X51_4,GTCAGCGAGTCTAGCT-1_10X51_4,ATTCCCGTCTTCCTAA-1_10X52_3,TGAATCGGTAGACGGT-1_10X51_3,TACGGTAGTAGGTACG-1_10X52_3,ATCCGTCCAACCGCTG-1_10X51_4,GCAGGCTAGACAAGCC-1_10X52_4,TTGCTGCTCTTTCTAG-1_10X51_4,GGATGTTCAGCGTACC-1_10X52_3,ATCAGGTAGGTCGTAG-1_10X52_4
0610007P14Rik,0,0,1,1,0,3,3,0,1,1,...,0,0,0,0,0,0,1,0,0,0
0610009B22Rik,1,1,0,1,0,0,0,1,0,1,...,0,0,0,0,1,0,0,0,0,1
0610009L18Rik,0,1,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
0610009O20Rik,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,0,1,0
0610010F05Rik,1,1,0,0,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Zyg11b,2,1,0,1,0,3,1,0,1,0,...,2,0,0,1,0,1,0,0,0,0
Zzef1,0,0,0,1,0,3,0,2,0,1,...,0,0,1,0,0,1,0,0,1,0
Zzz3,0,1,0,0,1,2,0,0,0,1,...,0,0,0,3,0,0,1,0,1,0
l7Rn6,3,0,0,0,1,0,0,0,1,1,...,0,0,1,0,0,0,0,0,1,0


In [215]:
cell_class = 'Vglut2'
delta_data_folder = '/bigdata/isaac/'+ cell_class + '_files/sex_stats/'+run+'/gene_delta_plots/data/'
genes = ['Ar', 'Esr1', 'Pgr']



# Change the current working directory to the specified path
os.chdir(delta_data_folder)

# Get a list of all files in the directory that include '_expr_mlog_df_c_'
raw_files = [f for f in os.listdir() if '_expr_raw_df_c_' in f]
metadata_files= [f for f in os.listdir() if '_metadata_df_c_' in f]
# Print the list of files
print ('raw files/metadata files')
print(raw_files)
print(metadata_files)

def extract_number(filename):
    match = re.search(r'-(\d+)-', filename)
    if match:
        return int(match.group(1))
    return float('inf')  # In case no number is found, place it at the end


# Sort the lists based on the extracted number
sorted_raw_files = sorted(raw_files, key=extract_number)
sorted_metadata_files = sorted(metadata_files, key=extract_number)

print ('sorted raw files/metadata files')
print(sorted_raw_files)
print(sorted_metadata_files)


for gene in genes:
    gene_expr_over_ct_df = pd.DataFrame(columns=['ct_id','Naïve-F (mean)', 'Naïve-F (std)', 'Breeder-F (mean)', 'Breeder-F (std)', 'Naïve-M (mean)', 'Naïve-M (std)', 'Breeder-M (mean)', 'Breeder-M (std)'])

    for r_file, m_file in zip(sorted_raw_files, sorted_metadata_files):
        #print(r_file)
        #print(m_file)
        raw_df = pd.read_json(r_file)
        metadata_df = pd.read_json(m_file)
        ct_id = (r_file.split('-')[1])
        if (np.all(raw_df.columns == metadata_df.columns)):
            if gene in raw_df.index:
                N_f_expr = np.mean(raw_df.loc[gene,metadata_df.loc['Group',:]=='Naïve-F'])
                N_f_std = np.std(raw_df.loc[gene,metadata_df.loc['Group',:]=='Naïve-F'])
                B_f_expr = np.mean(raw_df.loc[gene,metadata_df.loc['Group',:]=='Breeder-F'])
                B_f_std = np.std(raw_df.loc[gene,metadata_df.loc['Group',:]=='Breeder-F'])  
                N_m_expr = np.mean(raw_df.loc[gene,metadata_df.loc['Group',:]=='Naïve-M'])
                N_m_std = np.std(raw_df.loc[gene,metadata_df.loc['Group',:]=='Naïve-M'])
                B_m_expr = np.mean(raw_df.loc[gene,metadata_df.loc['Group',:]=='Breeder-M'])
                B_m_std = np.std(raw_df.loc[gene,metadata_df.loc['Group',:]=='Breeder-M'])
                tmp_arr = np.array([ct_id,N_f_expr, N_f_std, B_f_expr, B_f_std, N_m_expr, N_m_std, B_m_expr, B_m_std])
                tmp_arr = np.reshape(tmp_arr, (1,len(tmp_arr)))
                tmp = pd.DataFrame(tmp_arr,columns=gene_expr_over_ct_df.columns)
                gene_expr_over_ct_df = pd.concat([gene_expr_over_ct_df,tmp], axis=0)
            else:
                print ('Gene not found in raw data: ', gene, r_file)
        else: 
            print('Error: Column names do not match: ', r_file, m_file)
            break

    # Define the new directory name
    gene_ct_dir = 'gene_expr_over_ct'
    # Create the new directory
    gene_ct_dir_path = os.path.join('/bigdata/isaac/'+ cell_class + '_files/sex_stats/'+run, gene_ct_dir)
    os.makedirs(gene_ct_dir_path, exist_ok=True)

    # Convert 'ct_id' column to numeric
    gene_expr_over_ct_df['ct_id'] = pd.to_numeric(gene_expr_over_ct_df['ct_id'], errors='coerce')
    gene_expr_over_ct_df['Naïve-F (mean)'] = pd.to_numeric(gene_expr_over_ct_df['Naïve-F (mean)'], errors='coerce')
    gene_expr_over_ct_df['Naïve-F (std)'] = pd.to_numeric(gene_expr_over_ct_df['Naïve-F (std)'], errors='coerce')
    gene_expr_over_ct_df['Breeder-F (mean)'] = pd.to_numeric(gene_expr_over_ct_df['Breeder-F (mean)'], errors='coerce')
    gene_expr_over_ct_df['Breeder-F (std)'] = pd.to_numeric(gene_expr_over_ct_df['Breeder-F (std)'], errors='coerce')
    gene_expr_over_ct_df['Naïve-M (mean)'] = pd.to_numeric(gene_expr_over_ct_df['Naïve-M (mean)'], errors='coerce')
    gene_expr_over_ct_df['Naïve-M (std)'] = pd.to_numeric(gene_expr_over_ct_df['Naïve-M (std)'], errors='coerce')
    gene_expr_over_ct_df['Breeder-M (mean)'] = pd.to_numeric(gene_expr_over_ct_df['Breeder-M (mean)'], errors='coerce')
    gene_expr_over_ct_df['Breeder-M (std)'] = pd.to_numeric(gene_expr_over_ct_df['Breeder-M (std)'], errors='coerce')


    # Drop rows with NaN values in 'ct_id' column
    #gene_expr_over_ct_df = gene_expr_over_ct_df.dropna(subset=['ct_id'])
    #print (gene_expr_over_ct_df['ct_id'])

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.errorbar(np.array(gene_expr_over_ct_df['ct_id']), gene_expr_over_ct_df['Naïve-F (mean)'], yerr=gene_expr_over_ct_df['Naïve-F (std)'], fmt='o', label='Naïve-F', capsize=5)
    ax.errorbar(np.array(gene_expr_over_ct_df['ct_id']), gene_expr_over_ct_df['Breeder-F (mean)'], yerr=gene_expr_over_ct_df['Breeder-F (std)'], fmt='o', label='Breeder-F', capsize=5)
    ax.errorbar(np.array(gene_expr_over_ct_df['ct_id']), gene_expr_over_ct_df['Naïve-M (mean)'], yerr=gene_expr_over_ct_df['Naïve-M (std)'], fmt='o', label='Naïve-M', capsize=5)
    ax.errorbar(np.array(gene_expr_over_ct_df['ct_id']), gene_expr_over_ct_df['Breeder-M (mean)'], yerr=gene_expr_over_ct_df['Breeder-M (std)'], fmt='o', label='Breeder-M', capsize=5)

    ax.set_xticks(np.arange(1,int(ct_id)+1))
    ax.set_title(gene + ' Expression Over ' + cell_class + ' CT')
    ax.set_xlabel('CT')
    ax.set_ylabel('Mean Expression')
    ax.legend()
    plt.savefig(gene_ct_dir_path + '/'+ cell_class + '_gene_expr_over_ct_per_pop_' + gene + '.png')
    plt.show()

raw files/metadata files
['Vglut2_expr_raw_df_c_Vglut2-2-Synpr-Sox6.json', 'Vglut2_expr_raw_df_c_Vglut2-10-Lamb3-Col18a1.json', 'Vglut2_expr_raw_df_c_Vglut2-13-Tshz1-Calcr.json', 'Vglut2_expr_raw_df_c_Vglut2-9-Epha4-Penk.json', 'Vglut2_expr_raw_df_c_Vglut2-3-Tshz1-Tgfb2.json', 'Vglut2_expr_raw_df_c_Vglut2-1-Trh-Tes.json', 'Vglut2_expr_raw_df_c_Vglut2-14-Zfhx4-Nr4a2.json', 'Vglut2_expr_raw_df_c_Vglut2-15-Eomes-Tnfaip8.json', 'Vglut2_expr_raw_df_c_Vglut2-6-Otp-Zic5.json', 'Vglut2_expr_raw_df_c_Vglut2-5-Otp-Peg10.json', 'Vglut2_expr_raw_df_c_Vglut2-17-Otp-Avp.json', 'Vglut2_expr_raw_df_c_Vglut2-4-Otp-Crabp1.json', 'Vglut2_expr_raw_df_c_Vglut2-8-Arhgef40-Thrsp.json', 'Vglut2_expr_raw_df_c_Vglut2-7-Otp-Prlr.json', 'Vglut2_expr_raw_df_c_Vglut2-11-Tac1-Cartpt.json', 'Vglut2_expr_raw_df_c_Vglut2-12-Tshz1-Grp.json']
['Vglut2_metadata_df_c_Vglut2-8-Arhgef40-Thrsp.json', 'Vglut2_metadata_df_c_Vglut2-12-Tshz1-Grp.json', 'Vglut2_metadata_df_c_Vglut2-14-Zfhx4-Nr4a2.json', 'Vglut2_metadata_df_c_Vglut

<IPython.core.display.Javascript object>

Gene not found in raw data:  Esr1 Vglut2_expr_raw_df_c_Vglut2-1-Trh-Tes.json
Gene not found in raw data:  Esr1 Vglut2_expr_raw_df_c_Vglut2-3-Tshz1-Tgfb2.json
Gene not found in raw data:  Esr1 Vglut2_expr_raw_df_c_Vglut2-9-Epha4-Penk.json
Gene not found in raw data:  Esr1 Vglut2_expr_raw_df_c_Vglut2-10-Lamb3-Col18a1.json
Gene not found in raw data:  Esr1 Vglut2_expr_raw_df_c_Vglut2-11-Tac1-Cartpt.json
Gene not found in raw data:  Esr1 Vglut2_expr_raw_df_c_Vglut2-12-Tshz1-Grp.json
Gene not found in raw data:  Esr1 Vglut2_expr_raw_df_c_Vglut2-13-Tshz1-Calcr.json
Gene not found in raw data:  Esr1 Vglut2_expr_raw_df_c_Vglut2-14-Zfhx4-Nr4a2.json
Gene not found in raw data:  Esr1 Vglut2_expr_raw_df_c_Vglut2-15-Eomes-Tnfaip8.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [228]:
ct_id

'17'

In [223]:
r_file

'Vglut2_expr_raw_df_c_Vglut2-17-Otp-Avp.json'

In [225]:
# load the dataset
flowers = sns.load_dataset('iris') 

In [226]:
flowers

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica


In [233]:
flowersm = pd.melt(flowers, id_vars=["species"])

In [234]:
flowersm

,species,variable,value
0,setosa,sepal_length,5.1
1,setosa,sepal_length,4.9
2,setosa,sepal_length,4.7
3,setosa,sepal_length,4.6
4,setosa,sepal_length,5.0
...,...,...,...
595,virginica,petal_width,2.3
596,virginica,petal_width,1.9
597,virginica,petal_width,2.0
598,virginica,petal_width,2.3


In [ ]:
groups = np.unique(metadata_df.T['Group'])

In [222]:
gene

'Pgr'

In [230]:
raw_df.T[gene].loc[metadata_df.T['Group'] == 'Naïve-F']

AGGGTCCTCCGGTTCT-1_10X35_2    0
AAGGAATAGTTACTCG-1_10X35_1    0
AGGGTTTAGTCAGGGT-1_10X35_1    0
GTTGTGATCCGTGTGG-1_10X35_1    1
TGTACAGCAGCACACC-1_10X35_1    0
CTCCGATGTCATTGCA-1_10X35_1    0
ACGTTCCGTGTCACAT-1_10X35_2    0
TTTCAGTTCCCGTTGT-1_10X38_1    0
ACTATTCGTGTGCCTG-1_10X38_1    0
TGGATGTCAATTTCTC-1_10X35_2    0
Name: Pgr, dtype: int64

In [255]:
raw_df.T[gene]

AGGGTCCTCCGGTTCT-1_10X35_2    0
AAGGAATAGTTACTCG-1_10X35_1    0
AGGGTTTAGTCAGGGT-1_10X35_1    0
GTTGTGATCCGTGTGG-1_10X35_1    1
TGTACAGCAGCACACC-1_10X35_1    0
                             ..
AGCGTCGGTATCAGGG-1_10X51_3    0
AACAACCCAAACCACT-1_10X51_4    0
AAGTTCGGTCACAGTT-1_10X51_3    0
TTGAGTGTCGGCCTTT-1_10X52_4    0
CATGCCTTCTGTCTCG-1_10X51_3    0
Name: Pgr, Length: 107, dtype: int64

In [254]:
metadata_df.T['Group']

AGGGTCCTCCGGTTCT-1_10X35_2      Naïve-F
AAGGAATAGTTACTCG-1_10X35_1      Naïve-F
AGGGTTTAGTCAGGGT-1_10X35_1      Naïve-F
GTTGTGATCCGTGTGG-1_10X35_1      Naïve-F
TGTACAGCAGCACACC-1_10X35_1      Naïve-F
                                ...    
AGCGTCGGTATCAGGG-1_10X51_3    Breeder-M
AACAACCCAAACCACT-1_10X51_4    Breeder-M
AAGTTCGGTCACAGTT-1_10X51_3    Breeder-M
TTGAGTGTCGGCCTTT-1_10X52_4    Breeder-M
CATGCCTTCTGTCTCG-1_10X51_3    Breeder-M
Name: Group, Length: 107, dtype: object

In [229]:
metadata_df.T['Group'] == 'Naïve-F'

AGGGTCCTCCGGTTCT-1_10X35_2     True
AAGGAATAGTTACTCG-1_10X35_1     True
AGGGTTTAGTCAGGGT-1_10X35_1     True
GTTGTGATCCGTGTGG-1_10X35_1     True
TGTACAGCAGCACACC-1_10X35_1     True
                              ...  
AGCGTCGGTATCAGGG-1_10X51_3    False
AACAACCCAAACCACT-1_10X51_4    False
AAGTTCGGTCACAGTT-1_10X51_3    False
TTGAGTGTCGGCCTTT-1_10X52_4    False
CATGCCTTCTGTCTCG-1_10X51_3    False
Name: Group, Length: 107, dtype: bool

In [232]:
groups = np.unique(metadata_df.T['Group'])

In [236]:
for g in groups:
    print (g)
    print (np.where(metadata_df.T['Group'] == g))

Breeder-F
(array([10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26,
       27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43,
       44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60,
       61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77,
       78, 79]),)
Breeder-M
(array([ 91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106]),)
Naïve-F
(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),)
Naïve-M
(array([80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90]),)


In [10]:
%%capture output
cell_class = 'Nonneuronal'
run = '040325_run' # '311224_run' for gaba, Vglut1, Vglut2, use '170224_run' for NN
delta_data_folder = '/bigdata/isaac/'+ cell_class + '_files/sex_stats/'+run+'/gene_delta_plots/data/'
genes = ['Ar', 'Esr1', 'Pgr']

# Change the current working directory to the specified path
os.chdir(delta_data_folder)

# Get a list of all files in the directory that include '_expr_mlog_df_c_'
raw_files = [f for f in os.listdir() if '_expr_raw_df_c_' in f]
metadata_files= [f for f in os.listdir() if '_metadata_df_c_' in f]
# Print the list of files
print ('raw files/metadata files')
print(raw_files)
print(metadata_files)

def extract_number(filename):
    match = re.search(r'-(\d+)-', filename)
    if match:
        return int(match.group(1))
    return float('inf')  # In case no number is found, place it at the end


# Sort the lists based on the extracted number
sorted_raw_files = sorted(raw_files, key=extract_number)
sorted_metadata_files = sorted(metadata_files, key=extract_number)

print ('sorted raw files/metadata files')
print(sorted_raw_files)
print(sorted_metadata_files)


for gene in genes:

    df_merged_all = pd.DataFrame(columns=['gene', 'Group', 'ct_id', 'cell_class'])
    # Define the new directory name
    gene_ct_dir = 'gene_expr_over_ct_dodgepts_barmean'
    # Create the new directory
    gene_ct_dir_path = os.path.join('/bigdata/isaac/'+ cell_class + '_files/sex_stats/'+run, gene_ct_dir)
    os.makedirs(gene_ct_dir_path, exist_ok=True)
    
    for r_file, m_file in zip(sorted_raw_files, sorted_metadata_files):
        #print(r_file)
        #print(m_file)
        raw_df = pd.read_json(r_file)
        metadata_df = pd.read_json(m_file)
        ct_id = (r_file.split('-')[1])
        if (np.all(raw_df.columns == metadata_df.columns)):
                if gene in raw_df.index:
                    df_merged = pd.concat([raw_df.T[gene], metadata_df.T['Group']], axis=1)
                    df_merged.rename(columns={gene: 'gene'}, inplace=True)
                    df_merged['ct_id'] = ct_id
                    df_merged['cell_class'] = cell_class
        else:
            print('Error: Column names do not match: ', r_file, m_file)
        
        df_merged_all = pd.concat([df_merged_all, df_merged], axis=0)

        fig,ax = plt.subplots(figsize=(8, 4))
        p = sns.stripplot(data=df_merged_all, x="ct_id", y="gene", hue="Group", dodge=True, size=2)
        ax.set_title(gene + ' Expression Over ' + cell_class + ' CT')
        ax.set_xlabel('CT')
        ax.set_ylabel('Gene Normalized Expression')
        sns.pointplot(data=df_merged_all, x="ct_id", y="gene", hue="Group", dodge=0.532, join=False, palette="dark", markers="_", scale=0.75, errorbar=None)
        for x in range(0, len(df_merged_all['ct_id'].unique())):
            plt.axvspan(x - 0.5, x + 0.5, facecolor='black', alpha=[0.2 if x%2 == 0 else 0.1][0])
        plt.legend(bbox_to_anchor=(1.05, 1), loc=2)
        plt.tight_layout()
        plt.savefig(gene_ct_dir_path + '/'+ cell_class + '_dodgepts_barmean_' + gene + '.pdf')
        plt.show()
        

In [280]:
df_merged_all

,gene,Group,ct_id,cell_class
CCGAACGGTGATATAG-1_10X38_2,0,Naïve-F,1,Vglut2
AACCAACAGGATTTCC-1_10X35_1,0,Naïve-F,1,Vglut2
TCGACCTCAAAGGGCT-1_10X35_2,1,Naïve-F,1,Vglut2
GAACGTTCAAGAAACT-1_10X35_1,2,Naïve-F,1,Vglut2
TCCTCCCCAGGTGAGT-1_10X35_1,1,Naïve-F,1,Vglut2
...,...,...,...,...
AGCGTCGGTATCAGGG-1_10X51_3,0,Breeder-M,17,Vglut2
AACAACCCAAACCACT-1_10X51_4,0,Breeder-M,17,Vglut2
AAGTTCGGTCACAGTT-1_10X51_3,0,Breeder-M,17,Vglut2
TTGAGTGTCGGCCTTT-1_10X52_4,0,Breeder-M,17,Vglut2


In [262]:
df_merged = pd.concat([raw_df.T[gene], metadata_df.T['Group']], axis=1)
df_merged

,Pgr,Group
AGGGTCCTCCGGTTCT-1_10X35_2,0,Naïve-F
AAGGAATAGTTACTCG-1_10X35_1,0,Naïve-F
AGGGTTTAGTCAGGGT-1_10X35_1,0,Naïve-F
GTTGTGATCCGTGTGG-1_10X35_1,1,Naïve-F
TGTACAGCAGCACACC-1_10X35_1,0,Naïve-F
...,...,...
AGCGTCGGTATCAGGG-1_10X51_3,0,Breeder-M
AACAACCCAAACCACT-1_10X51_4,0,Breeder-M
AAGTTCGGTCACAGTT-1_10X51_3,0,Breeder-M
TTGAGTGTCGGCCTTT-1_10X52_4,0,Breeder-M


In [265]:
df_merged.rename(columns={gene: 'gene'}, inplace=True)
df_merged

,gene,Group
AGGGTCCTCCGGTTCT-1_10X35_2,0,Naïve-F
AAGGAATAGTTACTCG-1_10X35_1,0,Naïve-F
AGGGTTTAGTCAGGGT-1_10X35_1,0,Naïve-F
GTTGTGATCCGTGTGG-1_10X35_1,1,Naïve-F
TGTACAGCAGCACACC-1_10X35_1,0,Naïve-F
...,...,...
AGCGTCGGTATCAGGG-1_10X51_3,0,Breeder-M
AACAACCCAAACCACT-1_10X51_4,0,Breeder-M
AAGTTCGGTCACAGTT-1_10X51_3,0,Breeder-M
TTGAGTGTCGGCCTTT-1_10X52_4,0,Breeder-M


In [266]:
df_merged['ct_id'] = ct_id
df_merged['cell_class'] = cell_class
df_merged

,gene,Group,ct_id,cell_class
AGGGTCCTCCGGTTCT-1_10X35_2,0,Naïve-F,17,Vglut2
AAGGAATAGTTACTCG-1_10X35_1,0,Naïve-F,17,Vglut2
AGGGTTTAGTCAGGGT-1_10X35_1,0,Naïve-F,17,Vglut2
GTTGTGATCCGTGTGG-1_10X35_1,1,Naïve-F,17,Vglut2
TGTACAGCAGCACACC-1_10X35_1,0,Naïve-F,17,Vglut2
...,...,...,...,...
AGCGTCGGTATCAGGG-1_10X51_3,0,Breeder-M,17,Vglut2
AACAACCCAAACCACT-1_10X51_4,0,Breeder-M,17,Vglut2
AAGTTCGGTCACAGTT-1_10X51_3,0,Breeder-M,17,Vglut2
TTGAGTGTCGGCCTTT-1_10X52_4,0,Breeder-M,17,Vglut2


In [293]:
fig,ax = plt.subplots(figsize=(8, 4))
p = sns.stripplot(data=df_merged_all, x="ct_id", y="gene", hue="Group", dodge=True, size=2)
ax.set_title(gene + ' Expression Over ' + cell_class + ' CT')
ax.set_xlabel('CT')
ax.set_ylabel('Gene Normalized Expression')
sns.pointplot(data=df_merged_all, x="ct_id", y="gene", hue="Group", dodge=0.532, join=False, palette="dark", markers="_", scale=0.75, errorbar=None)
for x in range(0, len(df_merged_all['ct_id'].unique())):
    plt.axvspan(x - 0.5, x + 0.5, facecolor='black', alpha=[0.2 if x%2 == 0 else 0.1][0])
plt.legend(bbox_to_anchor=(1.05, 1), loc=2)
plt.tight_layout()
plt.savefig(gene_ct_dir_path + '/'+ cell_class + '_dodgepts_barmean_' + gene + '.pdf')
plt.show()

<IPython.core.display.Javascript object>

/tmp/ipykernel_2772882/977668759.py:6: UserWarning: 

The `scale` parameter is deprecated and will be removed in v0.15.0. You can now control the size of each plot element using matplotlib `Line2D` parameters (e.g., `linewidth`, `markersize`, etc.).

  sns.pointplot(data=df_merged_all, x="ct_id", y="gene", hue="Group", dodge=0.532, join=False, palette="dark", markers="_", scale=0.75, errorbar=None)
/tmp/ipykernel_2772882/977668759.py:6: UserWarning: 

The `join` parameter is deprecated and will be removed in v0.15.0. You can remove the line between points with `linestyle='none'`.

  sns.pointplot(data=df_merged_all, x="ct_id", y="gene", hue="Group", dodge=0.532, join=False, palette="dark", markers="_", scale=0.75, errorbar=None)


In [259]:
cell_class

'Vglut2'